# 33 — Federated learning: FedAvg in one scalar

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/omarahmedZa/ai-ml-course/blob/main/notebooks/033-federated-setting-fedavg.ipynb)

Three phones each hold a different number of examples and each would pull the model to a different value. Federated averaging trains locally, then combines the copies on a server. You will compute that combination by hand, run six rounds, and see which clients the final model actually serves.

## 1. The numbers: three clients, one average

Each client stores its own examples, so each client has a different idea of the right model. Here the model is a single number `w`, and a client's ideal value is the average target in its own data. The server must merge those three wishes into one number.

In [ ]:
import numpy as np                                  # the array library, preinstalled on Colab

sizes = np.array([100, 200, 700])                   # examples held by clients 1, 2 and 3
targets = np.array([0.1, 0.5, 0.9])                 # the w each client would pick alone
shares = sizes / sizes.sum()                        # each client's share of all the data
fedavg = float(shares @ targets)                    # weighted average: 0.74
plain = float(targets.mean())                       # equal average: 0.50

print("examples per client:", sizes)                # [100 200 700]
print("share of the data:", shares)                 # [0.1 0.2 0.7]
print("FedAvg global w:", round(fedavg, 3))         # 0.74
print("equal-weight global w:", round(plain, 3))    # 0.50
print(f"client 3 holds {shares[2]:.0%} of the data")   # 70%

**Line by line**

- `import numpy as np` — loads NumPy, which Colab already has.
- `sizes = ...` — how many examples each of the three clients owns.
- `targets = ...` — the model value each client would choose on its own.
- `shares = ...` — divide by the total, giving each client's fraction of the data.
- `fedavg = ...` — the weighted mean; `@` multiplies then sums.
- `plain = ...` — the unweighted mean, for comparison.
- `print("examples per client"...)` — how lopsided the split is.
- `print("share of the data"...)` — the three weights the server will use.
- `print("FedAvg global w"...)` — the answer FedAvg proposes.
- `print("equal-weight global w"...)` — the answer a naive server proposes.
- `print(f"client 3 holds ...")` — the f-string formats a fraction as a percent.

## 2. Step by step: six rounds of FedAvg

A round has two halves. Every client takes a gradient step on its own data, then the server replaces all copies with their weighted average. Starting from zero, follow the number that the clients agree on.

In [ ]:
w = 0.0                                             # the server's starting model
local = np.zeros(3)                                 # one copy of the model per client
lr = 0.5                                            # size of one local gradient step
history = []                                        # global w after every round

for r in range(6):                                  # six communication rounds
    local = local - lr * (local - targets)          # each client steps toward its own target
    w = float(shares @ local)                       # server: average, weighted by data
    history.append(w)                               # remember the global model
    print(f"round {r+1}: clients {np.round(local, 2)} -> global {w:.3f}")   # readout
    local = np.full(3, w)                           # broadcast the global model back

print("final global w:", round(w, 3))               # 0.728, still climbing to 0.74

**Line by line**

- `w = 0.0` — the server has no information yet, so it starts at zero.
- `local = np.zeros(3)` — three copies, one sent to each client.
- `lr = 0.5` — how far one local step moves the model.
- `history = []` — an empty list; we append the global value each round.
- `for r in range(6)` — six rounds of local work plus one server average.
- `local = local - lr * (local - targets)` — one gradient step for all clients at once.
- `w = float(shares @ local)` — the server's weighted average of the three copies.
- `history.append(w)` — keep the number for the plot in section 3.
- `print(f"round ...")` — shows the client copies, then the new global value.
- `local = np.full(3, w)` — every client restarts from the same averaged model.
- `print("final global w"...)` — the value the rounds settled on.

## 3. Plot: where the global model lands

The curve shows the global model after each round. The green line is the weighted mean of the three targets and the red dotted line is the equal mean; the gap between them is the cost of ignoring how much data each client holds.

In [ ]:
import matplotlib.pyplot as plt                     # plotting library, preinstalled on Colab

rounds = np.arange(1, len(history) + 1)             # x-axis: round numbers 1 to 6
plt.figure(figsize=(6, 3))                          # a wide, short figure
plt.plot(rounds, history, "o-", color="tab:blue", label="global w")        # the trajectory
plt.axhline(fedavg, color="tab:green", linestyle="--", label="weighted 0.74")   # target
plt.axhline(plain, color="tab:red", linestyle=":", label="equal 0.50")          # shortcut
plt.ylim(0, 1)                                      # room above and below the values
plt.xlabel("communication round")                   # x-axis label
plt.ylabel("global w")                              # y-axis label
plt.legend(loc="lower right")                       # name the three lines
plt.title("FedAvg climbs to the data-weighted mean")   # the takeaway
plt.show()                                          # render the figure

**Line by line**

- `import matplotlib.pyplot as plt` — loads the plotting library.
- `rounds = ...` — one x value per entry in `history`.
- `plt.figure(figsize=(6, 3))` — a wide, short canvas.
- `plt.plot(...)` — the global model against the round number.
- `plt.axhline(fedavg, ...)` — a horizontal line at the weighted mean.
- `plt.axhline(plain, ...)` — a second line at the equal mean.
- `plt.ylim(0, 1)` — both lines and the curve stay visible.
- `plt.xlabel(...)` — names the horizontal axis.
- `plt.ylabel(...)` — names the vertical axis.
- `plt.legend(...)` — a key for the curve and the two lines.
- `plt.title(...)` — states what the reader should conclude.
- `plt.show()` — draws the finished figure.

## 4. Interactive: local epochs on a slider

One local gradient step per round is a choice, not a law. Drag the slider to give every client several steps between averages and watch the curve reach its endpoint sooner. The title reports the final global value.

In [ ]:
from ipywidgets import interact                      # sliders that rerun a function

def run(epochs=1):                                   # epochs = local steps per round
    w = 0.0                                          # server starts from zero
    local = np.zeros(3)                              # three client copies
    path = []                                        # global w per round
    for r in range(6):                               # same six rounds as before
        for e in range(epochs):                      # extra local work per round
            local = local - lr * (local - targets)   # each client moves toward its target
        w = float(shares @ local)                    # weighted server average
        local = np.full(3, w)                        # broadcast the average back
        path.append(w)                               # record this round
    plt.figure(figsize=(6, 3))                       # fresh figure on every drag
    plt.plot(range(1, 7), path, "o-", color="tab:purple")   # trajectory
    plt.axhline(fedavg, color="grey", linestyle="--")       # the weighted mean
    plt.ylim(0, 1)                                   # fixed axes, so nothing jumps
    plt.xlabel("communication round")                # x-axis label
    plt.ylabel("global w")                           # y-axis label
    plt.title(f"{epochs} local epoch(s): final w = {path[-1]:.3f}")   # live readout
    plt.show()                                       # draw it

interact(run, epochs=(1, 5, 1))                      # slider: 1 to 5 local epochs

**Line by line**

- `from ipywidgets import interact` — Colab ships the widgets library.
- `def run(epochs=1)` — the function that redraws one setting.
- `w = 0.0` — every drag restarts the server from zero.
- `local = np.zeros(3)` — a fresh copy per client.
- `path = []` — collects this run's global values.
- `for r in range(6)` — the outer loop over communication rounds.
- `for e in range(epochs)` — the inner loop over local steps.
- `local = ...` — the same gradient step, repeated `epochs` times.
- `w = ...` — the server averages after the local work.
- `local = np.full(3, w)` — the broadcast that ends the round.
- `path.append(w)` — store the value for the line plot.
- `plt.figure(...)` — a new figure for each slider position.
- `plt.plot(...)` — the global value per round.
- `plt.axhline(...)` — a reference line at the weighted mean.
- `plt.ylim(0, 1)` — fixed axes, so nothing jumps.
- `plt.xlabel("communication round")` — names the horizontal axis.
- `plt.ylabel("global w")` — names the vertical axis.
- `plt.title(...)` — the live readout of epochs and final value.
- `plt.show()` — draw the frame.
- `interact(run, epochs=(1, 5, 1))` — the slider and its range.

## 5. When it breaks: equal weights ignore 70% of the data

Replace the weighted average with a plain mean and the 700-example client becomes a minority in its own federation. The bars compare each client's squared error under the two servers; the big client's error jumps sixfold.

In [ ]:
import numpy as np                                # reload the array library
import matplotlib.pyplot as plt                   # reload the plotting library

err_fed = (fedavg - targets) ** 2                 # squared error per client under FedAvg
err_eq = (plain - targets) ** 2                   # same, under equal weighting
loss_fed = float(shares @ err_fed)                # data-weighted total loss, FedAvg
loss_eq = float(shares @ err_eq)                  # data-weighted total loss, equal
print("weighted loss, FedAvg:", round(loss_fed, 4))    # 0.0704
print("weighted loss, equal :", round(loss_eq, 4))     # 0.1280
print("big client error, FedAvg:", round(float(err_fed[2]), 3))   # 0.026
print("big client error, equal :", round(float(err_eq[2]), 3))    # 0.160

idx = np.arange(3)                                # three x positions, one per client
plt.figure(figsize=(6, 3))                        # a wide, short figure
plt.bar(idx - 0.2, err_fed, width=0.4, label="FedAvg", color="tab:green")       # weighted
plt.bar(idx + 0.2, err_eq, width=0.4, label="equal", color="tab:red")           # unweighted
plt.xticks(idx, ["100 ex", "200 ex", "700 ex"])   # name the clients
plt.ylabel("squared error on that client")        # y-axis label
plt.legend()                                      # name the two bar groups
plt.title("equal weights blow up the 70% client") # the takeaway
plt.show()                                        # render the figure

**Line by line**

- `import numpy as np` — the array library, reloaded for this cell.
- `import matplotlib.pyplot as plt` — the plotting library.
- `err_fed = ...` — squared distance from each client's target to the FedAvg model.
- `err_eq = ...` — the same distances for the equally weighted model.
- `loss_fed = ...` — average those errors, weighting by data size.
- `loss_eq = ...` — the same weighted total for the equal model.
- `print(...)` — the two totals side by side.
- `print(...)` — the same comparison as a single number.
- `print("big client error, FedAvg"...)` — error on the 700-example client.
- `print("big client error, equal"...)` — the same client under equal weights.
- `idx = ...` — x positions shared by both bar groups.
- `plt.figure(...)` — a fresh canvas.
- `plt.bar(idx - 0.2, ...)` — green bars, shifted left.
- `plt.bar(idx + 0.2, ...)` — red bars, shifted right.
- `plt.xticks(...)` — replace positions with client names.
- `plt.ylabel(...)` — names the vertical axis.
- `plt.legend()` — a key for green versus red.
- `plt.title(...)` — the takeaway.
- `plt.show()` — draws the chart.

## Your turn

Change `sizes` in section 1 to `[400, 300, 300]` and predict the FedAvg value before rerunning. Then drag section 4 to five local epochs: does the endpoint change, or only the speed?